# Hands-on 29 (Bab 11): Mini Proyek Klasifikasi Sederhana di Raspberry Pi / Emulator

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 11.

Mini proyek end-to-end meniru deployment edge: simulator sensor -> model lokal
(joblib dari Hands-on 27) -> status klasifikasi -> log CSV dan/atau MQTT.

## Tujuan

Pada latihan ini, pembaca akan menyusun mini proyek end-to-end yang meniru deployment nyata di Raspberry Pi atau emulator Linux sejenis. Tujuannya:

- membaca data sensor atau simulasi sensor,
- menjalankan model lokal,
- menghasilkan status klasifikasi,
- dan, bila diinginkan, meneruskan hasil ke MQTT atau log file.

Latihan ini sangat penting karena menyatukan semua konsep:

- training di laptop
- ekspor model
- inferensi lokal
- integrasi edge ke pipeline IoT

Ini juga sangat sejalan dengan arsitektur edge yang dijelaskan dalam referensi, di mana edge node:

- menerima data dari device,
- menjalankan komputasi lokal,
- lalu mengirim hasil atau ringkasan ke lapisan lain.

## 7. Contoh pengembangan lanjutan

Dari mini proyek ini, pembaca bisa berkembang ke:

- threshold + model hybrid
- logging ke SQLite lokal
- kirim hasil ke dashboard Grafana
- retraining periodik di laptop lalu redeploy ke edge
- integrasi Raspberry Pi dengan sensor nyata
- percobaan TinyML yang jauh lebih kecil di ESP32

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

# --- Utilitas notebook: menjalankan skrip buku APA ADANYA (tanpa flag) ---
# Skrip buku memakai while True / loop_forever, jadi notebook menghentikannya
# dari luar setelah beberapa detik dan menyimpan keluaran parsialnya.
import subprocess, sys, time
from pathlib import Path

LOG_DIR = Path("outputs/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def latar(script, nama_log, args=()):
    """Jalankan skrip buku di latar belakang (python -u), keluaran ke file log."""
    f = open(LOG_DIR / nama_log, "w", encoding="utf-8")
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=f, stderr=subprocess.STDOUT)
    return p, f

def henti(*pasangan):
    """Hentikan proses latar belakang dan tutup file lognya."""
    for p, f in pasangan:
        p.terminate()
        try:
            p.wait(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
        f.close()

def jalankan_buku(script, detik, args=()):
    """Jalankan skrip buku selama `detik` detik lalu hentikan; kembalikan keluaran."""
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        out, _ = p.communicate(timeout=detik)
    except subprocess.TimeoutExpired:
        p.terminate()
        try:
            out, _ = p.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
            out, _ = p.communicate()
    return out

def log_teks(nama_log):
    return (LOG_DIR / nama_log).read_text(encoding="utf-8")


[INFO] Working directory: /downloads/iot-ml-book


## 1. Mini Proyek tanpa MQTT (simulator -> model -> log CSV)

In [2]:
print(jalankan_buku("scripts/edge_room_classifier_project.py", 90))

=== Mini proyek edge room classifier dimulai ===
step=0 | T=32.18 H=54.73 CO2=620 P=149.0 -> status=warning
step=1 | T=26.33 H=44.3 CO2=1482 P=279.15 -> status=fault
step=2 | T=26.11 H=68.75 CO2=560 P=126.81 -> status=normal
step=3 | T=30.71 H=49.79 CO2=474 P=253.97 -> status=warning
step=4 | T=25.3 H=52.49 CO2=742 P=187.19 -> status=normal
step=5 | T=32.0 H=56.79 CO2=1083 P=234.89 -> status=warning
step=6 | T=26.23 H=47.58 CO2=1042 P=142.92 -> status=warning
step=7 | T=27.79 H=47.8 CO2=859 P=163.91 -> status=normal
step=8 | T=24.77 H=58.38 CO2=1353 P=184.33 -> status=normal
step=9 | T=25.06 H=69.16 CO2=578 P=205.1 -> status=normal
step=10 | T=28.36 H=40.37 CO2=1167 P=142.49 -> status=warning
step=11 | T=26.1 H=41.79 CO2=1212 P=118.42 -> status=normal
step=12 | T=28.67 H=50.23 CO2=819 P=275.54 -> status=normal
step=13 | T=31.95 H=42.09 CO2=838 P=210.02 -> status=warning
step=14 | T=27.91 H=50.61 CO2=1429 P=240.7 -> status=normal
step=15 | T=30.46 H=44.68 CO2=632 P=128.77 -> status=warn

## 2. Verifikasi Log CSV

In [3]:
import pandas as pd
log = pd.read_csv("outputs/reports/edge_room_predictions.csv")
print(log.head(10).to_string(index=False))
print("\nDistribusi status:", log["predicted_status"].value_counts().to_dict())

 step  temperature_c  humidity_pct  co2_ppm  power_watt predicted_status
    0          32.18         54.73      620      149.00          warning
    1          26.33         44.30     1482      279.15            fault
    2          26.11         68.75      560      126.81           normal
    3          30.71         49.79      474      253.97          warning
    4          25.30         52.49      742      187.19           normal
    5          32.00         56.79     1083      234.89          warning
    6          26.23         47.58     1042      142.92          warning
    7          27.79         47.80      859      163.91           normal
    8          24.77         58.38     1353      184.33           normal
    9          25.06         69.16      578      205.10           normal

Distribusi status: {'normal': 14, 'warning': 13, 'fault': 3}


## 3. Versi dengan MQTT (broker lokal)

In [4]:
import json
import paho.mqtt.subscribe as mqttsub
broker = subprocess.Popen([sys.executable, "scripts/mqtt/run_broker.py"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
try:
    # Subscriber penerima prediksi (skrip buku scripts/mqtt/subscribe_sensor.py memakai topik room01/telemetry,
    # jadi di sini dipakai paho.mqtt.subscribe untuk topik prediksi), lalu skrip edge buku dijalankan apa adanya.
    import threading
    hasil = {}
    def dengar():
        hasil["msgs"] = mqttsub.simple("iotlab/room01/prediction", hostname="127.0.0.1", port=1883, qos=1, msg_count=6)
    t = threading.Thread(target=dengar); t.start()
    time.sleep(1)
    out = jalankan_buku("scripts/edge_room_classifier_mqtt.py", 30)
    print(out)
    ringkas = __import__("re").findall(r"'predicted_status': '(\w+)'", out)
    print("Jumlah publish:", len(ringkas), "| distribusi:", dict(__import__("collections").Counter(ringkas)))
    t.join(timeout=10)
    print("--- Pesan prediksi yang diterima subscriber ---")
    for m in hasil.get("msgs", []):
        print(m.topic, m.payload.decode())
finally:
    broker.terminate()

Published: {'step': 0, 'temperature_c': 30.43, 'humidity_pct': 44.05, 'co2_ppm': 647, 'power_watt': 235.57, 'predicted_status': 'warning'}
Published: {'step': 1, 'temperature_c': 29.43, 'humidity_pct': 57.48, 'co2_ppm': 460, 'power_watt': 235.73, 'predicted_status': 'warning'}
Published: {'step': 2, 'temperature_c': 31.01, 'humidity_pct': 46.11, 'co2_ppm': 813, 'power_watt': 205.84, 'predicted_status': 'warning'}
Published: {'step': 3, 'temperature_c': 31.24, 'humidity_pct': 49.56, 'co2_ppm': 1186, 'power_watt': 117.88, 'predicted_status': 'warning'}
Published: {'step': 4, 'temperature_c': 31.9, 'humidity_pct': 69.82, 'co2_ppm': 837, 'power_watt': 105.64, 'predicted_status': 'warning'}
Published: {'step': 5, 'temperature_c': 32.47, 'humidity_pct': 68.14, 'co2_ppm': 967, 'power_watt': 215.65, 'predicted_status': 'warning'}
Published: {'step': 6, 'temperature_c': 32.86, 'humidity_pct': 54.3, 'co2_ppm': 1251, 'power_watt': 215.37, 'predicted_status': 'fault'}
Published: {'step': 7, 'tempe

## Panduan Lengkap dari Buku

_Bagian ini dikutip dari buku PDF (sumber kebenaran praktikum) dan diformat ulang agar mudah dibaca. Kode asli buku tidak diubah; bagian instalasi/persiapan lingkungan diringkas karena sudah dijelaskan di README._

### 2. Target perangkat

Raspberry Pi

Untuk buku ini, Raspberry Pi adalah target paling realistis karena: ●   menjalankan Linux ●   bisa memakai Python ●   mudah integrasi dengan MQTT dan file lokal ●   cocok sebagai edge gateway praktis

Emulator

Jika pembaca belum punya Raspberry Pi, maka: ●   laptop Ubuntu ●   VM Linux ●   mini PC ●   atau container lokal dapat dipakai sebagai emulator logika edge Pendekatan ini sangat realistis dan sesuai semangat buku yang fokus pada praktik cepat, bukan menunggu perangkat ideal.

### 3. Struktur mini proyek

Kita buat alur sederhana:

simulator sensor → model lokal → hasil status → log / MQTT

### 4. Source code mini proyek tanpa MQTT

Buat file scripts/edge_room_classifier_project.py:

```python
from pathlib import Path
import time
import random
import joblib
import pandas as pd
MODEL_PATH = Path("models/room_status_rf.joblib")
OUTPUT_LOG = Path("outputs/reports/edge_room_predictions.csv")
OUTPUT_LOG.parent.mkdir(parents=True, exist_ok=True)
model = joblib.load(MODEL_PATH)
records = []
def build_feature_row(temp, hum, co2, power, hour, dow):
    # contoh fitur sederhana; pada implementasi nyata
    # beberapa lag/rolling sebaiknya dipertahankan dari buffer lokal
    row = {
        "temperature_c": temp,
        "humidity_pct": hum,
        "co2_ppm": co2,
        "power_watt": power,
        "hour": hour,
        "day_of_week": dow,
        "is_weekend": int(dow >= 5),
        "temp_lag_1": temp,
        "co2_lag_1": co2,
        "power_lag_1": power,
        "temp_ma_3": temp,
        "co2_ma_3": co2,
        "power_ma_3": power
    }
    return pd.DataFrame([row])
print("=== Mini proyek edge room classifier dimulai ===")
for step in range(30):
    temp = round(random.uniform(24.0, 33.0), 2)
    hum = round(random.uniform(40.0, 70.0), 2)
    co2 = random.randint(450, 1500)
    power = round(random.uniform(100.0, 300.0), 2)
    hour = (8 + step // 3) % 24
     dow = 2   # misal Rabu
     X = build_feature_row(temp, hum, co2, power, hour, dow)
     pred = model.predict(X)[0]
    print(
        f"step={step} | T={temp} H={hum} CO2={co2} P={power} ->
status={pred}"
    )
     records.append({
         "step": step,
         "temperature_c": temp,
         "humidity_pct": hum,
         "co2_ppm": co2,
         "power_watt": power,
         "predicted_status": pred
     })
     time.sleep(0.3)
pd.DataFrame(records).to_csv(OUTPUT_LOG, index=False)
print("Log hasil inferensi disimpan ke:", OUTPUT_LOG)
```

Jalankan:

```python
python scripts/edge_room_classifier_project.py
```

### 5. Versi mini proyek dengan MQTT opsional

Jika pembaca ingin alur yang lebih “IoT banget”, kita tambahkan MQTT.

Buat file scripts/edge_room_classifier_mqtt.py:

```python
from pathlib import Path
import json
import time
import random
import joblib
import pandas as pd
import paho.mqtt.client as mqtt
MODEL_PATH = Path("models/room_status_rf.joblib")
TOPIC = "iotlab/room01/prediction"
model = joblib.load(MODEL_PATH)
client = mqtt.Client(client_id="edge-room-classifier")
client.connect("localhost", 1883, 60)
def build_feature_row(temp, hum, co2, power, hour, dow):
    row = {
        "temperature_c": temp,
        "humidity_pct": hum,
        "co2_ppm": co2,
        "power_watt": power,
        "hour": hour,
         "day_of_week": dow,
         "is_weekend": int(dow >= 5),
         "temp_lag_1": temp,
         "co2_lag_1": co2,
         "power_lag_1": power,
         "temp_ma_3": temp,
         "co2_ma_3": co2,
         "power_ma_3": power
    }
    return pd.DataFrame([row])
for step in range(20):
    temp = round(random.uniform(24.0, 33.0), 2)
    hum = round(random.uniform(40.0, 70.0), 2)
    co2 = random.randint(450, 1500)
    power = round(random.uniform(100.0, 300.0), 2)
    hour = (8 + step // 3) % 24
    dow = 2
    X = build_feature_row(temp, hum, co2, power, hour, dow)
    pred = model.predict(X)[0]
    payload = {
        "step": step,
        "temperature_c": temp,
        "humidity_pct": hum,
        "co2_ppm": co2,
        "power_watt": power,
        "predicted_status": pred
    }
    client.publish(TOPIC, json.dumps(payload), qos=1)
    print("Published:", payload)
    time.sleep(0.5)
client.disconnect()
```

Uji subscriber cepat

```python
       mosquitto_sub -h localhost -t "iotlab/room01/prediction" -v
Dengan ini, mini proyek edge sudah benar-benar menyentuh pola
operasional IoT:
    ● akuisisi data
    ● inferensi lokal
    ● pengiriman hasil
```

### 6. Apa yang membuat proyek ini “realistis”?

Karena alurnya sama dengan banyak sistem edge nyata:

- model dilatih di tempat lain
- edge hanya menerima model final
- edge menerima data sensor
- edge membuat keputusan lokal
- hasil disimpan atau dikirim

Ini konsisten dengan pembagian peran antara cloud/laptop dan edge yang dijelaskan dalam referensi edge computing.

## Langkah Kerja dari Buku

### 8. Langkah kerja yang harus dilakukan pembaca

Tahap 1

Latih model di laptop Ubuntu.

Tahap 2

Ekspor model: ● joblib untuk model klasik ● .tflite untuk model Keras ringan

Tahap 3

Uji inferensi lokal satu per satu.

Tahap 4

Simulasikan aliran data seolah-olah dari sensor.

Tahap 5

Jalankan mini proyek di Raspberry Pi atau emulator Linux.

Tahap 6

Tambahkan output: ●   terminal ●   CSV ●   MQTT ●   atau dashboard

Tahap 7

Evaluasi apakah model: ●   cukup cepat ●   cukup stabil ●   cukup ringan ●   dan layak untuk edge

## Penutup dari Buku



## Penutup

Mini proyek menyatukan seluruh konsep Bab 11: training di laptop, ekspor model,
inferensi lokal per sampel, lalu hasil diteruskan ke log CSV dan/atau broker MQTT.
Prediksi yang diterbitkan ke topik `iotlab/room01/prediction` terverifikasi diterima
subscriber. Log simulasi tersimpan di `outputs/reports/edge_room_predictions.csv`.